In [ ]:
from common.spark_session import spark

In [ ]:
catalog = "sandbox-rey-01"
schemas = ["bronze", "silver", "gold"]
externalLocation = "loc_sandbox_sblakera"

In [ ]:
baseUrl = spark.sql(f"DESCRIBE EXTERNAL LOCATION `{externalLocation}`").select("URL").collect()[0][0]
paths = {schema: f"{baseUrl}medallion/{schema}" for schema in schemas}

processedPaths = {}

for schema, path in paths.items():
    processedPaths[schema] = path

print(processedPaths)

In [ ]:
def createSchema(catalog, path, schema):
    print(f"""Using {catalog} """)
    spark.sql(f""" USE CATALOG `{catalog}`""")
    print(f"""Creating {schema} Schema in {catalog}""")
    spark.sql(f"""CREATE SCHEMA IF NOT EXISTS `{schema}` MANAGED LOCATION '{path}'""")

In [ ]:
for schema, processedPath in processedPaths.items():
    createSchema(catalog, processedPath, schema)

In [ ]:
def createTableRawTraffic(catalog):
    print(f'Creating raw_Traffic table in {catalog}')
    spark.sql(f"""CREATE TABLE IF NOT EXISTS `{catalog}`.`bronze`.`raw_traffic`
                        (
                            Record_ID INT,
                            Count_point_id INT,
                            Direction_of_travel VARCHAR(255),
                            Year INT,
                            Count_date VARCHAR(255),
                            hour INT,
                            Region_id INT,
                            Region_name VARCHAR(255),
                            Local_authority_name VARCHAR(255),
                            Road_name VARCHAR(255),
                            Road_Category_ID INT,
                            Start_junction_road_name VARCHAR(255),
                            End_junction_road_name VARCHAR(255),
                            Latitude DOUBLE,
                            Longitude DOUBLE,
                            Link_length_km DOUBLE,
                            Pedal_cycles INT,
                            Two_wheeled_motor_vehicles INT,
                            Cars_and_taxis INT,
                            Buses_and_coaches INT,
                            LGV_Type INT,
                            HGV_Type INT,
                            EV_Car INT,
                            EV_Bike INT,
                            Extract_Time TIMESTAMP
                    );""")
    
    print("************************************")

In [ ]:
def createTableRawRoad(catalog):
    print(f'Creating raw_roads table in {catalog}')
    spark.sql(f"""CREATE TABLE IF NOT EXISTS `{catalog}`.`bronze`.`raw_roads`
                        (
                            Road_ID INT,
                            Road_Category_Id INT,
                            Road_Category VARCHAR(255),
                            Region_ID INT,
                            Region_Name VARCHAR(255),
                            Total_Link_Length_Km DOUBLE,
                            Total_Link_Length_Miles DOUBLE,
                            All_Motor_Vehicles DOUBLE
                    );""")
    
    print("************************************")

In [ ]:
createTableRawTraffic(catalog)
createTableRawRoad(catalog)